# ANALIZA REZULTATA DOBIJENIH KORIŠĆENJEM ISKLJUČIVO STFT

## Učitavanje dataset-a

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('./results.csv')
df.head()

,song_name,genre,duration [s],sample_rate,processing_time_demucs,bpm_song,bpm_drums,bpm_hpss,key_song,confidence_song,key_demucs,confidence_demucs,key_hpss,confidence_hpss,manual_bpm,manual_key,agreement_bpm,agreement_key
0,Eric Clapton - Tears In Heaven.wav,acoustic,274.95,44100,154.23,152.00,152.00,152.00,A Major,0.6694,A Major,0.7211,A Major,0.6648,77,A Major,0.00,3
1,Extreme - More Than Words.wav,acoustic,255.56,44100,137.62,92.29,46.14,92.29,F# Major,0.6938,F# Major,0.3700,F# Major,0.6802,92,F# Major,21.75,3
2,Oasis - Wonderwall.wav,acoustic,279.50,44100,149.61,117.45,117.45,117.45,E Major,0.7251,E Major,0.6343,E Major,0.7387,87,F# Minor,0.00,3
3,The Beatles - Blackbird.wav,acoustic,138.41,44100,82.67,93.96,93.96,93.96,G Minor,0.3990,G Minor,0.3579,G Minor,0.3962,94,G Major,0.00,3
4,Tracy Chapman - Fast Car.wav,acoustic,266.26,44100,178.10,103.36,103.36,103.36,A Minor,0.4734,A Major,0.3703,A Minor,0.4777,102,A Major,0.00,2


## BPM Error (ne uzimajući u obzir Half-time i Double-time)

In [3]:
def bpm_error(predicted, original):
    candidates = [predicted, predicted * 2, predicted / 2]
    errors = [abs(c - original) for c in candidates]

    return min(errors)

In [4]:
df['error_song'] = df.apply(lambda row: bpm_error(row['bpm_song'], row['manual_bpm']), axis=1)
df['error_drums'] = df.apply(lambda row: bpm_error(row['bpm_drums'], row['manual_bpm']), axis=1)
df['error_hpss'] = df.apply(lambda row: bpm_error(row['bpm_hpss'], row['manual_bpm']), axis=1)

## BPM MAE

In [5]:
mae_song = df['error_song'].mean()
mae_drums = df['error_drums'].mean()
mae_hpss = df['error_hpss'].mean()

print("BPM MAE")
print("=" * 20)
print(f'Song MAE: {round(mae_song, 2)}')
print(f'Drums MAE: {round(mae_drums, 2)}')
print(f'HPSS MAE: {round(mae_hpss, 2)}')

BPM MAE
Song MAE: 5.71
Drums MAE: 4.35
HPSS MAE: 4.48


Kao i što je očekivano, BPM MAE je najmanja za **izolovane bubnjeve korišćenjem Demucs modela**.

Medjutim, i **HPSS metoda** daje vrlo dobar rezultat, dok je BPM MAE **na celoj pesmi** znatno veća u odnosu na prethodne dve metode.

## BPM Accuracy

In [6]:
TOLERANCE = 5

In [7]:
df["song_bpm_correct"] = (df["error_song"] <= TOLERANCE)
df["drums_bpm_correct"] = (df["error_drums"] <= TOLERANCE)
df["hpss_bpm_correct"] = (df["error_hpss"] <= TOLERANCE)

In [8]:
song_bpm_acc = (df["song_bpm_correct"].mean() * 100)
drums_bpm_acc = (df["drums_bpm_correct"].mean() * 100)
hpss_bpm_acc = (df["hpss_bpm_correct"].mean() * 100)

print(f"[BPM]: Song Accuracy:  {round(song_bpm_acc, 2)}%")
print(f"[BPM]: Drums Accuracy: {round(drums_bpm_acc, 2)}%")
print(f"[BPM]: HPSS Accuracy:  {round(hpss_bpm_acc, 2)}%")

[BPM]: Song Accuracy:  81.82%
[BPM]: Drums Accuracy: 86.36%
[BPM]: HPSS Accuracy:  86.36%


## Key Accuracy

In [9]:
df["song_key_correct"] = (df["key_song"]==df["manual_key"])
df["demucs_key_correct"] = (df["key_demucs"]==df["manual_key"])
df["hpss_key_correct"] = (df["key_hpss"]==df["manual_key"])

song_key_acc = (df["song_key_correct"].mean() * 100)
demucs_key_acc = (df["demucs_key_correct"].mean() * 100)
hpss_key_acc = (df["hpss_key_correct"].mean() * 100)

print(f"Song Key Accuracy:   {round(song_key_acc, 2)}%")
print(f"Demucs Accuracy:     {round(demucs_key_acc, 2)}%")
print(f"HPSS Accuracy:       {round(hpss_key_acc, 2)}%")

Song Key Accuracy:   47.27%
Demucs Accuracy:     53.64%
HPSS Accuracy:       47.27%


Detekcija tonaliteta je najbolja na izolovanim instrumentima dobijenim Demucs metodom. Medjutim, sam rezultat od 53.64% nije dobar, ali nije ni loš uzimajući u obzir žanrove sa specifičnim karakteristikama.

Analizom je uočeno da Song i HPSS metoda daju identične rezultate.

## Confidence analiza

In [10]:
bins = [0, 0.4, 0.6, 0.8, 1.0]

labels = [
    "0-0.4",
    "0.4-0.6",
    "0.6-0.8",
    "0.8-1.0"
]

In [11]:
df["conf_group"] = pd.cut(df["confidence_demucs"], bins=bins, labels=labels)

In [12]:
confidence_analysis = (df.groupby("conf_group")["demucs_key_correct"].mean() * 100)
print(confidence_analysis)

conf_group
0-0.4      46.153846
0.4-0.6    50.000000
0.6-0.8    55.357143
0.8-1.0    60.000000
Name: demucs_key_correct, dtype: float64


## Key Accuracy po žanru

In [13]:
genre_key = (
    df.groupby("genre")["demucs_key_correct"]
    .mean()
    * 100
).sort_values(ascending=False)

genre_key

genre
folk          100.0
disco          80.0
electronic     80.0
reggae         80.0
rap            80.0
acoustic       60.0
country        60.0
blues          60.0
hiphop         60.0
kpop           60.0
rock           60.0
techno         60.0
metal          60.0
trance         60.0
house          40.0
rnb            40.0
pop            40.0
latin          40.0
punk           20.0
funk           20.0
soul           20.0
jazz            0.0
Name: demucs_key_correct, dtype: float64

Upravo se ovde može uočiti da se najveće greške dešavaju kod žanrova sa specifičnim karakteristikama.

In [14]:
summary = pd.DataFrame({

    "Method": [
        "Song",
        "Demucs",
        "HPSS"
    ],

    "BPM MAE": [
        mae_song,
        mae_drums,
        mae_hpss
    ],

    "BPM Accuracy": [
        song_bpm_acc,
        drums_bpm_acc,
        hpss_bpm_acc
    ],

    "Key Accuracy": [
        song_key_acc,
        demucs_key_acc,
        hpss_key_acc
    ]
})

summary

,Method,BPM MAE,BPM Accuracy,Key Accuracy
0,Song,5.711682,81.818182,47.272727
1,Demucs,4.348727,86.363636,53.636364
2,HPSS,4.479955,86.363636,47.272727


## Provera kada je samo detekcija na pesmi ili samo detekcija na Demucs-u ili samo detekcija pomoću HPSS-a tačna

In [15]:
df["only_song_correct"] = (
    df["song_key_correct"]
    &
    ~df["demucs_key_correct"]
    &
    ~df["hpss_key_correct"]
)

df["only_demucs_correct"] = (
    df["demucs_key_correct"]
    &
    ~df["song_key_correct"]
    &
    ~df["hpss_key_correct"]
)

df["only_hpss_correct"] = (
    df["hpss_key_correct"]
    &
    ~df["song_key_correct"]
    &
    ~df["demucs_key_correct"]
)

print("Samo nad pesmom tačno:", df["only_song_correct"].sum())
print("Samo nad Demucs-om tačno:", df["only_demucs_correct"].sum())
print("Samo nad HPSS tačno:", df["only_hpss_correct"].sum())

Samo nad pesmom tačno: 0
Samo nad Demucs-om tačno: 16
Samo nad HPSS tačno: 0


Ovde potvrdjujemo da Song i HPSS metoda daju indentične rezultate.

## Provera kada su sva tri načina detekcije tonaliteta dovele to tačnog rezultata

In [16]:
df["all_correct"] = (
    df["song_key_correct"]
    &
    df["demucs_key_correct"]
    &
    df["hpss_key_correct"]
)

print(f"Sve tri metode tačne: {df["all_correct"].sum()}/{df.shape[0]}")

Sve tri metode tačne: 41/110


## Provera kada su sva tri načina detekcije tonaliteta dovele do pogrešnog rezultata

In [17]:
df["all_wrong"] = (
    ~df["song_key_correct"]
    &
    ~df["demucs_key_correct"]
    &
    ~df["hpss_key_correct"]
)

print(f"Sve tri metode pogrešile: {df["all_wrong"].sum()}/{df.shape[0]}")

Sve tri metode pogrešile: 41/110


Broj pesama gde su sve tri metode pogrešile u detekciji tonaliteta je 41 od 110, što predstavlja veliki broj pesama (oko 35%).

## Glasanje izmedju metoda kako bi se dobio tačan rezultat (Confidence weighted voting)

Uvodjenje metode glasanja korišćenjem confidence kolone.

In [18]:
from collections import defaultdict

def confidence_vote(row):

    votes = defaultdict(float)

    votes[row["key_song"]] += row["confidence_song"]
    votes[row["key_demucs"]] += row["confidence_demucs"]
    votes[row["key_hpss"]] += row["confidence_hpss"]

    return max(
        votes,
        key=votes.get
    )

In [19]:
df["key_confidence_vote"] = df.apply(
    confidence_vote,
    axis=1
)

In [20]:
df["confidence_vote_correct"] = (
    df["key_confidence_vote"]
    ==
    df["manual_key"]
)

accuracy = (
    df["confidence_vote_correct"]
    .mean()
    * 100
)

print(accuracy)

50.0


Broj od 50% za metodu glasanja nije čudan, upravo iz razloga što Song i HPSS metoda daju identične rezultate i tako "nadglašavaju" Demucs model. Medjutim, ni confidence nije validna mera za ovu metodu, jer je uočeno da model iako pogreši, daje veliki confidence.

In [21]:
key_comparison = pd.DataFrame({

    "Method": [
        "Song",
        "Demucs",
        "HPSS",
        "Confidence weighted voting"
    ],

    "Accuracy": [

        df["song_key_correct"]
        .mean() * 100,

        df["demucs_key_correct"]
        .mean() * 100,

        df["hpss_key_correct"]
        .mean() * 100,

        df["confidence_vote_correct"]
        .mean() * 100
    ]
})

key_comparison

,Method,Accuracy
0,Song,47.272727
1,Demucs,53.636364
2,HPSS,47.272727
3,Confidence weighted voting,50.000000


## Pronalazak pesama gde su sve tri metode pogresno odredile tonalitet

In [22]:
all_wrong_df = df[
    df["all_wrong"]
]

all_wrong_df[
    [
        "genre",
        "song_name",
        "manual_key",
        "key_song",
        "key_demucs",
        "key_hpss"
    ]
]

,genre,song_name,manual_key,key_song,key_demucs,key_hpss
2,acoustic,Oasis - Wonderwall.wav,F# Minor,E Major,E Major,E Major
3,acoustic,The Beatles - Blackbird.wav,G Major,G Minor,G Minor,G Minor
5,blues,Albert King - Born Under A Bad Sign.wav,C# Minor,C# Major,G# Minor,C# Major
7,blues,Robert Johnson - Cross Road Blues.wav,B Major,A# Minor,B Minor,B Minor
11,country,Dolly Parton - Jolene.wav,C# Minor,C# Major,G# Minor,C# Major
12,country,John Denver - Take Me Home Country Roads.wav,A Major,E Major,E Major,E Major
16,disco,Bee Gees - Stayin Alive.wav,F Minor,F Major,F Major,F Major
23,electronic,deadmau5 - Strobe.wav,G# Minor,B Major,B Major,B Major
30,funk,Average White Band - Pick up the Pieces.wav,F Minor,D# Major,C Minor,D# Major
32,funk,Parliament - Give Up The Funk Tear The Roof Of...,E Minor,B Minor,B Minor,B Minor


In [23]:
all_wrong_df["genre"].value_counts()

genre
jazz          4
soul          4
funk          3
house         3
punk          3
rnb           3
acoustic      2
blues         2
country       2
hiphop        2
latin         2
techno        2
trance        2
disco         1
electronic    1
kpop          1
metal         1
pop           1
reggae        1
rock          1
Name: count, dtype: int64

# ANALIZA REZULTATA DOBIJENIH KORIŠĆENJEM STFT+CQT

**CQT (Constant-Q Transform)** je za razliku od STFT metoda koja više odgovara muzici, jer ima logaritamsku frekvencijsku skalu za razliku od STFT koja ima linearnu.

In [24]:
import pandas as pd
import numpy as np

df = pd.read_csv('./new_results.csv')
df.head()

,song_name,genre,duration[s],sample_rate,processing_time_demucs[s],bpm_song,bpm_drums,bpm_hpss,key_song,key_demucs,key_hpss,key_song_cqt,key_demucs_cqt,key_hpss_cqt,manual_bpm,manual_key,agreement_bpm,agreement_key
0,Eric Clapton - Tears In Heaven.wav,acoustic,274.95,44100,161.59,152.00,152.00,152.00,A Major,A Major,A Major,A Major,A Major,A Major,77,A Major,0.00,3
1,Extreme - More Than Words.wav,acoustic,255.56,44100,146.82,92.29,46.14,92.29,F# Major,F# Major,F# Major,F# Major,F# Major,F# Major,92,F# Major,21.75,3
2,Oasis - Wonderwall.wav,acoustic,279.50,44100,159.65,117.45,117.45,117.45,E Major,E Major,E Major,F# Minor,F# Minor,A Major,87,F# Minor,0.00,3
3,The Beatles - Blackbird.wav,acoustic,138.41,44100,78.89,93.96,93.96,93.96,G Minor,G Minor,G Minor,G Major,G Major,G Major,94,G Major,0.00,3
4,Tracy Chapman - Fast Car.wav,acoustic,266.26,44100,151.47,103.36,103.36,103.36,A Minor,A Major,A Minor,A Major,A Major,A Major,102,A Major,0.00,2


## BPM analiza

In [25]:
def bpm_error(predicted, manual):
    if pd.isna(manual):
        return np.nan

    candidates = [predicted, predicted * 2,predicted / 2]

    errors = [abs(c - manual) for c in candidates]

    return min(errors)

In [26]:
df["bpm_error_song"] = df.apply(
    lambda row: bpm_error(row["bpm_song"], row["manual_bpm"]), axis=1)

df["bpm_error_drums"] = df.apply(
    lambda row: bpm_error(row["bpm_drums"], row["manual_bpm"]), axis=1)

df["bpm_error_hpss"] = df.apply(lambda row: bpm_error(row["bpm_hpss"], row["manual_bpm"]), axis=1)

## BPM MAE

In [27]:
print("BPM MAE")
print("=" * 20)
print(f"Song: {round(df["bpm_error_song"].mean(), 2)}")
print(f"Drums: {round(df["bpm_error_drums"].mean(), 2)}")
print(f"HPSS: {round(df["bpm_error_hpss"].mean(), 2)}")

BPM MAE
Song: 5.71
Drums: 4.34
HPSS: 4.48


## BPM Accuracy (±5 BPM)

In [28]:
def bpm_accuracy(error_series):
    return ((error_series <= 5).mean() * 100)

print("BPM Accuracy")
print("=" * 20)
print(f"Song: {round(bpm_accuracy(df["bpm_error_song"]), 2)}%")
print(f"Drums: {round(bpm_accuracy(df["bpm_error_drums"]), 2)}%")
print(f"HPSS: {round(bpm_accuracy(df["bpm_error_hpss"]), 2)}%")

BPM Accuracy
Song: 81.82%
Drums: 86.36%
HPSS: 86.36%


## Analiza tonaliteta

In [29]:
def key_accuracy(column):
    return (df[column] == df["manual_key"]).mean() * 100

#### STFT Accuracy

In [30]:
print("STFT")
print("=" * 20)
print(f"Song: {round(key_accuracy("key_song"), 2)}%")
print(f"Demucs: {round(key_accuracy("key_demucs"), 2)}%")
print(f"HPSS: {round(key_accuracy("key_hpss"), 2)}%")

STFT
Song: 47.27%
Demucs: 53.64%
HPSS: 47.27%


#### CQT Accuracy

In [31]:
print("CQT")
print("=" * 20)
print(f"Song: {round(key_accuracy("key_song_cqt"), 2)}%")
print(f"Demucs: {round(key_accuracy("key_demucs_cqt"), 2)}%")
print(f"HPSS: {round(key_accuracy("key_hpss_cqt"), 2)}%")

CQT
Song: 55.45%
Demucs: 63.64%
HPSS: 53.64%


Ovde uočavamo znatno poboljšanje detekcije tonaliteta, gde se korišćenjem CQT i Demucs modela dobija accuracy od 63.64%.

## Poredjenje svih metoda

In [32]:
methods = [
    "key_song",
    "key_demucs",
    "key_hpss",

    "key_song_cqt",
    "key_demucs_cqt",
    "key_hpss_cqt"
]

In [33]:
results = []

for method in methods:
    accuracy = ((df[method] == df["manual_key"]).mean() * 100)
    results.append([method, accuracy])

accuracy_df = pd.DataFrame(
    results,
    columns=[
        "Method",
        "Accuracy"
    ]
)

accuracy_df.sort_values("Accuracy", ascending=False)

,Method,Accuracy
4,key_demucs_cqt,63.636364
3,key_song_cqt,55.454545
5,key_hpss_cqt,53.636364
1,key_demucs,53.636364
0,key_song,47.272727
2,key_hpss,47.272727


Ne samo da je CQT+Demucs bolji od STFT+Demucs, već je svaka metoda bolja u kombinaciji sa CQT nego sa STFT.

## Provera u koliko procenata je bar jedna metoda dala tačan rezultat

In [34]:
df["any_correct"] = (
    (df["key_song"] == df["manual_key"]) |
    (df["key_demucs"] == df["manual_key"]) |
    (df["key_hpss"] == df["manual_key"]) |
    (df["key_song_cqt"] == df["manual_key"]) |
    (df["key_demucs_cqt"] == df["manual_key"]) |
    (df["key_hpss_cqt"] == df["manual_key"])
)

print(f"{round(df["any_correct"].mean() * 100, 2)}%")

83.64%


Ovo predstavlja ogromno poboljšanje u odnosu na ranije rezultate koji su koristili isključivo STFT. Ovim smo pokazali da je u samo 16.36% slučajeva svaka metoda pogrešila tonalitet, što je mnogo bolje od prethodnog rezultata gde je taj procenat bio oko 35%.

## Provera da li je CQT stvarno i koliko bolji od STFT

#### Za celu pesmu

In [35]:
song_stft = (df["key_song"] == df["manual_key"])
song_cqt = (df["key_song_cqt"] == df["manual_key"])

print(f"CQT je na celoj pesmi pogodio {(song_cqt & ~song_stft).sum()} puta gde je STFT pogrešio.")

CQT je na celoj pesmi pogodio 27 puta gde je STFT pogrešio.


#### Za izolovane instrumente (Demucs)

In [36]:
demucs_stft = (df["key_demucs"] == df["manual_key"])
demucs_cqt = (df["key_demucs_cqt"] == df["manual_key"])

print(f"CQT je na izolovanim Demucs instrumentima pogodio {(demucs_cqt & ~demucs_stft).sum()} puta gde je STFT pogrešio.")

CQT je na izolovanim Demucs instrumentima pogodio 20 puta gde je STFT pogrešio.


#### Za izolovane instrumente+vokal (HPSS harmonics)

In [37]:
demucs_stft = (df["key_hpss"] == df["manual_key"])
demucs_cqt = (df["key_hpss_cqt"] == df["manual_key"])

print(f"CQT je na izolovanim HPSS harmonics pogodio {(demucs_cqt & ~demucs_stft).sum()} puta gde je STFT pogrešio.")

CQT je na izolovanim HPSS harmonics pogodio 24 puta gde je STFT pogrešio.


## Analiza po žanrovima

#### Analiza za BPM

In [38]:
genre_bpm = df.groupby("genre")[
    [
        "bpm_error_song",
        "bpm_error_drums",
        "bpm_error_hpss"
    ]
].mean()

genre_bpm.sort_values("bpm_error_drums")

,bpm_error_song,bpm_error_drums,bpm_error_hpss
genre,,,
house,0.108,0.108,0.108
reggae,0.384,0.384,0.384
kpop,0.454,0.454,0.454
disco,0.874,0.874,0.874
funk,0.890,0.890,0.890
electronic,0.940,0.940,0.940
rock,0.946,0.946,0.946
hiphop,5.017,0.966,0.951
techno,1.030,1.030,1.030


#### Analiza za tonalitet

Koristimo kolonu *key_demucs_cqt* jer ima najbolji accuracy.

In [39]:
genre_key = df.groupby("genre").apply(
    lambda x:
    (
        x["key_demucs_cqt"]
        ==
        x["manual_key"]
    ).mean() * 100
)

genre_key = genre_key.sort_values(
    ascending=False
)

genre_key

genre
acoustic      100.0
disco         100.0
folk          100.0
electronic    100.0
blues          80.0
reggae         80.0
rap            80.0
rock           80.0
country        60.0
jazz           60.0
soul           60.0
kpop           60.0
trance         60.0
techno         60.0
pop            40.0
metal          40.0
latin          40.0
house          40.0
funk           40.0
hiphop         40.0
rnb            40.0
punk           40.0
dtype: float64

Za razliku od metoda koje su koristile STFT, ova metoda nije ni za jedan žanr pogrešila u detekciji tonaliteta za svaku pesmu, čak šta više, za svaki žanr, ova metoda daje vrlo dobar rezultat.

## Pregled pesama kod kojih CQT "pobedjuje" STFT

#### Za celu pesmu

In [40]:
df[(df["key_song_cqt"] == df["manual_key"]) & (df["key_song"] != df["manual_key"])][
    [
        "song_name",
        "genre",
        "manual_key",
        "key_song",
        "key_song_cqt"
    ]
]

,song_name,genre,manual_key,key_song,key_song_cqt
2,Oasis - Wonderwall.wav,acoustic,F# Minor,E Major,F# Minor
3,The Beatles - Blackbird.wav,acoustic,G Major,G Minor,G Major
4,Tracy Chapman - Fast Car.wav,acoustic,A Major,A Minor,A Major
5,Albert King - Born Under A Bad Sign.wav,blues,C# Minor,C# Major,C# Minor
6,B.B. King - The Thrill Is Gone.wav,blues,B Minor,B Major,B Minor
7,Robert Johnson - Cross Road Blues.wav,blues,B Major,A# Minor,B Major
12,John Denver - Take Me Home Country Roads.wav,country,A Major,E Major,A Major
13,Johnny Cash - Ring of Fire.wav,country,G Major,D Major,G Major
16,Bee Gees - Stayin Alive.wav,disco,F Minor,F Major,F Minor
23,deadmau5 - Strobe.wav,electronic,G# Minor,B Major,G# Minor


#### Za Demucs

In [41]:
df[(df["key_demucs_cqt"] == df["manual_key"]) & (df["key_demucs"] != df["manual_key"])][
    [
        "song_name",
        "genre",
        "manual_key",
        "key_demucs",
        "key_demucs_cqt"
    ]
]

,song_name,genre,manual_key,key_demucs,key_demucs_cqt
2,Oasis - Wonderwall.wav,acoustic,F# Minor,E Major,F# Minor
3,The Beatles - Blackbird.wav,acoustic,G Major,G Minor,G Major
7,Robert Johnson - Cross Road Blues.wav,blues,B Major,B Minor,B Major
12,John Denver - Take Me Home Country Roads.wav,country,A Major,E Major,A Major
16,Bee Gees - Stayin Alive.wav,disco,F Minor,F Major,F Minor
23,deadmau5 - Strobe.wav,electronic,G# Minor,B Major,G# Minor
29,The Animals - House Of The Rising Sun.wav,folk,A Minor,E Major,A Minor
31,James Brown - Get Up Offa That Thing.wav,funk,E Minor,F# Minor,E Minor
42,Eric Prydz - Pjanoo.wav,house,G Minor,D Minor,G Minor
44,Swedish House Mafia ft. John Martin - Dont You...,house,B Minor,D Major,B Minor


#### Za HPSS

In [42]:
df[(df["key_hpss_cqt"] == df["manual_key"]) & (df["key_hpss"] != df["manual_key"])][
    [
        "song_name",
        "genre",
        "manual_key",
        "key_hpss",
        "key_hpss_cqt"
    ]
]

,song_name,genre,manual_key,key_hpss,key_hpss_cqt
3,The Beatles - Blackbird.wav,acoustic,G Major,G Minor,G Major
4,Tracy Chapman - Fast Car.wav,acoustic,A Major,A Minor,A Major
7,Robert Johnson - Cross Road Blues.wav,blues,B Major,B Minor,B Major
12,John Denver - Take Me Home Country Roads.wav,country,A Major,E Major,A Major
13,Johnny Cash - Ring of Fire.wav,country,G Major,D Major,G Major
16,Bee Gees - Stayin Alive.wav,disco,F Minor,F Major,F Minor
23,deadmau5 - Strobe.wav,electronic,G# Minor,B Major,G# Minor
25,Bob Dylan - Blowin in the Wind.wav,folk,D Major,A Major,D Major
34,Wild Cherry - Play That Funky.wav,funk,E Minor,E Major,E Minor
36,Kendrick Lamar - HUMBLE.wav,hiphop,C# Minor,E Minor,C# Minor


## Kada i gde sve metode greše?

In [43]:
all_wrong = df[
    (df["key_song"] != df["manual_key"]) &
    (df["key_demucs"] != df["manual_key"]) &
    (df["key_hpss"] != df["manual_key"]) &
    (df["key_song_cqt"] != df["manual_key"]) &
    (df["key_demucs_cqt"] != df["manual_key"]) &
    (df["key_hpss_cqt"] != df["manual_key"])
]

print(f"Sve metode su pogrešile u {len(all_wrong)} slučaja.")

Sve metode su pogrešile u 18 slučaja.


#### Provera po žanrovima gde su sve metode pogrešile

In [44]:
all_wrong["genre"].value_counts()

genre
funk       2
jazz       2
latin      2
soul       2
country    1
hiphop     1
house      1
kpop       1
metal      1
punk       1
reggae     1
rnb        1
rock       1
techno     1
Name: count, dtype: int64